In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dt = pd.read_csv('../data/processed/train_data.csv')

In [2]:
#Identify numeric columns
numeric_cols = dt.select_dtypes(include=['int64', 'float64']).columns

#Find potential categorical columns based on unique value count
potential_categorical = []
threshold = 100 

for col in numeric_cols:
    unique_count = dt[col].nunique()
    
    #Exclude the target variable from this list to prevent data leakage
    if unique_count < threshold and col != 'isFraud':
        potential_categorical.append(col)

print(f"Found {len(potential_categorical)} disguised categorical columns.")

#Convert them to 'category' data type
dt[potential_categorical] = dt[potential_categorical].astype('category')

print("Type casting completed. Memory usage optimized.")

Found 248 disguised categorical columns.
Type casting completed. Memory usage optimized.


In [3]:
v_138_166 = [f'V{i}' for i in range(138, 167)]
v_322_339 = [f'V{i}' for i in range(322, 340)]
id_series = [f'id_{str(i).zfill(2)}' for i in [24, 25, 7, 8, 21, 26, 27, 23, 22, 18, 4, 3, 33, 9, 10, 30, 32, 34, 14]]

other_cols = ['dist2', 'D7', 'D13', 'D14', 'D12', 'D6', 'D8', 'D9','addr2', 'DeviceInfo', 'TransactionID', 'TransactionDT']

high_null_cols = v_138_166 + v_322_339 + id_series + other_cols

dt['TransactionAmt_Log'] = np.log1p(dt['TransactionAmt'])

dt1 = dt.drop(columns=list(high_null_cols) + ['TransactionAmt'])

In [7]:
missing_values = dt1.isnull().mean() * 100

# Drop all columns with missing values > 30%
THRESHOLD = 30
columns_to_drop = missing_values[missing_values > THRESHOLD].index.tolist()
dt1 = dt1.drop(columns=columns_to_drop)

In [8]:
dt1.head(5)

,isFraud,ProductCD,card1,card2,card3,card4,card5,card6,addr1,P_emaildomain,...,V313,V314,V315,V316,V317,V318,V319,V320,V321,TransactionAmt_Log
0,0,W,13926,NaN,150.0,discover,142.0,credit,315.0,NaN,...,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,4.241327
1,0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,gmail.com,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.401197
2,0,W,4663,490.0,150.0,visa,166.0,debit,330.0,outlook.com,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.094345
3,0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,yahoo.com,...,0.0,0.0,0.0,50.0,1404.0,790.0,0.0,0.0,0.0,3.931826
4,0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,gmail.com,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.931826


In [9]:
# --- MISSING VALUE ANALYSIS ---
print("--- MISSING VALUE ANALYSIS ---")

# Calculate missing value percentages for all columns
missing_values = dt1.isnull().sum()
missing_percentages = (missing_values / len(dt1)) * 100

# Create a DataFrame for better viewing
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Missing_Percentage': missing_percentages})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {dt.shape[1]}")
print("\nTop 15 columns with most missing values:")
display(missing_df.head(15))

--- MISSING VALUE ANALYSIS ---
Total columns with missing values: 181 out of 435

Top 15 columns with most missing values:


,Missing_Count,Missing_Percentage
M6,169360,28.678836
V49,168969,28.612626
V50,168969,28.612626
V52,168969,28.612626
V51,168969,28.612626
V37,168969,28.612626
V38,168969,28.612626
V39,168969,28.612626
V40,168969,28.612626
V41,168969,28.612626


In [10]:
def auto_impute_missing_values(df):
    """
    Automated missing value imputation strategy (Max missing ratio ~29%):
    - Ratio < 10%: Simple Imputation (Median/Mode).
    - Ratio >= 10%: Impute (Median/Mode) + Create 'is_missing' indicator flag.
    """
    total_rows = len(df)
    new_indicator_columns = {}
    
    for col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_ratio = missing_count / total_rows
        
        # 1. Skip if no missing values
        if missing_ratio == 0:
            continue
            
        # 2. For ratios >= 10%, create an indicator flag to help the ML model
        if missing_ratio >= 0.10:
            new_indicator_columns[f'{col}_is_missing'] = df[col].isnull().astype(np.int8)
            
        # 3. Impute all missing values
        if df[col].dtype.name == 'category' or df[col].dtype == 'object':
            # Categorical: Fill with Mode
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
        else:
            # Numerical: Fill with Median
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            
    # Concat all indicator columns at once for better performance
    if new_indicator_columns:
        indicator_df = pd.DataFrame(new_indicator_columns)
        df = pd.concat([df, indicator_df], axis=1)
                
    return df

# Apply the function
dt1 = auto_impute_missing_values(dt1)
print(f"Imputation completed. New dataset shape: {dt1.shape}")

Imputation completed. New dataset shape: (590540, 288)


In [11]:
target_mean = dt1.groupby('P_emaildomain')['isFraud'].mean()
dt1['P_emaildomain'] = dt1['P_emaildomain'].map(target_mean).fillna(0)

dt1 = pd.get_dummies(dt1, columns=['card4', 'card6'], drop_first=True, dtype=int)

object_cols = dt1.select_dtypes(include=['object']).columns
dt1[object_cols] = dt1[object_cols].astype('category')

In [12]:
dt1.head()

,isFraud,ProductCD,card1,card2,card3,card5,addr1,P_emaildomain,C1,C2,...,V91_is_missing,V92_is_missing,V93_is_missing,V94_is_missing,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0,W,13926,361.0,150.0,142.0,315.0,0.039444,1.0,1.0,...,0,0,0,0,1,0,0,1,0,0
1,0,W,2755,404.0,150.0,102.0,325.0,0.039444,1.0,1.0,...,0,0,0,0,0,1,0,1,0,0
2,0,W,4663,490.0,150.0,166.0,330.0,0.094584,1.0,1.0,...,0,0,0,0,0,0,1,0,1,0
3,0,W,18132,567.0,150.0,117.0,476.0,0.022757,2.0,5.0,...,0,0,0,0,0,1,0,0,1,0
4,0,H,4497,514.0,150.0,102.0,420.0,0.039444,1.0,1.0,...,1,1,1,1,0,1,0,1,0,0


In [13]:
dt1.to_csv('../data/processed/train_cleaned.csv', index=False)